## Stream Customers Data From Cloud Files to Delta Lake using Auto Loader
1. Read files from cloud storage using Auto Loader
1. Transform the dataframe to add the following columns
    -   file path: Cloud file path
    -   ingestion date: Current Timestamp
1. Write the transformed data stream to Delta Lake Table



### 1. Read files using Auto Loader

###Auto Loader
Auto Loader in Databricks is a powerful feature designed to efficiently and incrementally load data from cloud storage into Databricks, especially when new files keep arriving continuously. It's ideal for building streaming data pipelines without manually tracking new files.

In [0]:
#spark.readStream.format("cloudFiles") - Initializes Auto Loader for incremental file ingestion from cloud storage
customer_df=(
  spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/myspace/streaming/autoloder/_schema")
  .load('/Volumes/workspace/default/myspace/streaming/autoloder/')
)

In [0]:
customer_df=(
  spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/myspace/streaming/autoloder/_schema")
  .option('cloudFiles.inferColumnTypes', 'true')
  .load('/Volumes/workspace/default/myspace/streaming/autoloder/')
)

When incoming JSON data contains fields that don't match your defined or inferred schema, Databricks automatically captures those "orphaned" fields in the _rescued_data column as JSON strings, rather than dropping them or causing the stream to fail.


In [0]:
customer_df=(
  spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/myspace/streaming/autoloder/_schema")
  .option('cloudFiles.inferColumnTypes', 'true')
  .option('cloudFiles.schemaHints', 'date_of_birth date,  member_since date, created_timestamp timestamp')
  .load('/Volumes/workspace/default/myspace/streaming/autoloder/')
)

### 2. Transform the dataframe to add the following columns
- file path: Cloud file path
- ingestion date: Current Timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, col
 
customers_transformed_df = (
    customer_df
    .withColumn('file_path', col('_metadata.file_path'))
    .withColumn('ingestion_date', current_timestamp())
)
 

### 3. Write the transformed data stream to Delta Table 

In [0]:

streaming_query=(
  customers_transformed_df.writeStream
  .format("delta")
  .option("checkpointLocation", "/Volumes/workspace/default/myspace/streaming/autoloder/_checkpoints1")
  .option("mergeSchema", "true")
  .trigger(once=True)
  .toTable("p101.p101schema.customers_autoloader")
)

In [0]:
#streaming_query.stop()

In [0]:
%sql
SELECT * FROM p101.p101schema.customers_autoloader;